# 08. Vehicle-parameter mismatch grid for PID+FF and the linear swing laws (arms 1 and 2)

**MRAC-RL logbook, step 3 of 4.** The offline comparison that Javier asked for after the professor meeting (2026-11-24): the **same controllers**, designed for the nominal vehicle, run on a plant whose **vehicle** parameters differ (mass, inertia, arm, motor asymmetry), one parameter at a time, at the nominal load (4 kg, 2 m). The three controllers are the mass-blind PID+FF (arm 1), the PID+FF with the softer law $G(-1,-0.5)$ and with the damper $G(-1,-2)$, both swing laws in the **fixed form**.

> **EXPLORATORY / offline / PROXY.** Reduced taut point-load plant (`QuadLoadPend`), development PC; the plant mismatch is `CanonicalOffline.with_vehicle_mismatch` (`experiments/mrac/baselines.py`, W2d), the controllers keep their nominal values. Offline metrics are named `c_T_off`, `c_S_off`: **not** the canonical `c_T`, `c_S`. One deterministic run per cell: no seeds, no intervals, no stability claim, no claim about the Pymunk chain plant. No MRAC arm is in this grid (it waits for the nominal-payload reference of notebook 07).

### Notation used across this series (added 2026-11-25)

The thesis notation (`thesis_book/residual.tex`, lines 95-292) is primary. The code and the derivation document (MQP) use their own symbols, mapped in the thesis file `.ai/research/derivations/notation-residual-tex.md`.

| This notebook / code | residual.tex | Note |
|---|---|---|
| `m_L`, `l` | $m_b$, $\ell$ | payload mass, rope length |
| `m_q`, `J`; `f`, `tau` | $m_q$, $J_q$; $f_q$, $\tau_q$ | |
| `phi` (code) | $\varphi$ | **$\varphi_{\rm tex} = -\varphi_{\rm code}$** with the cable at the vehicle CoG ($d = 0$). Odd functions of $\varphi$ flip sign; $\cos\varphi$, the tension and the tangent do not. With an offset $d\ne0$ the map is nonlinear |
| task reference `p_r(t)` | $x^d$ ("reference state" in residual.tex) | not Guha's reference model $x_r$ |

Numbers in this notebook are unchanged; only the reading of the symbols is aligned.

## 1. Research question

1. **Sensitivity.** How much does the tracking and swing performance of the nominal-vehicle PID+FF change when the true vehicle mass $m_q$, inertia $J$, motor arm $a_p$ or motor effectiveness asymmetry $\delta$ differs from the value the controller and allocator were designed for?
2. **Swing laws under mismatch.** Do the softer law and the damper keep their swing reduction relative to the mass-blind PID+FF when the vehicle is mismatched, and what do they cost in tracking?
3. **Reference point.** What is the mismatch baseline that a later arm 3 (the A0 law on a nominal-payload reference) has to be compared against, with the same cases, scenarios and metrics?

## 2. Hypotheses and predicted direction

Written from the mechanisms of section 4. **Honesty note:** the grid had been computed once, to check the pipeline, before this text was finalised, so these are not blind predictions; they were not changed after looking at the numbers, and each is checked by a computed cell in section 6 (a hypothesis may fail).

| # | Hypothesis | Predicted direction |
|---|---|---|
| H1 | The nominal case (all scales 1, $\delta=0$) reproduces the notebook-06 grid at $(L,m_L)=(2\text{ m},4\text{ kg})$ for the three controllers, and every ratio against the nominal vehicle is exactly 1 there | identity (same code path, same plant) |
| H2 | Motor asymmetry $\delta$ is the most damaging family for tracking: for every controller and scenario its worst `c_T_off_vs_nom` (at $\delta=0.10$) exceeds the worst case of $m_q$, $J$ and $a_p$, because the allocator splits the thrust equally and an uncorrected torque bias results | `c_T_off_vs_nom` $>1$, increasing with $\delta$; $\delta$ the largest family |
| H3 | A vehicle-mass mismatch of either sign raises `c_T_off_vs_nom` above 1 (the PID gravity compensation uses the nominal mass), increasing with $|\Delta m_q|$ on each side | `c_T_off_vs_nom` $>1$ for $\pm10$ and $\pm20$ %, larger at 20 % |
| H4 | Inertia and arm mismatch in the stated moderate ranges change tracking less than $m_q$ and $\delta$ do: for every controller and scenario the largest $|$`c_T_off_vs_nom`$-1|$ over the $J$ and $a_p$ cases is below the smallest over the $m_q=\pm10$ % and $\delta=0.05$ cases | smaller effect (how much smaller is read off section 6, not predicted): both change only the gain of the attitude loop |
| H5 | The softer law and the damper keep reducing the swing relative to the mass-blind PID+FF in every mismatch case and scenario (no sign reversal), because the swing law acts on the measured cable angle and the cable is not mismatched | `c_S_off_vs_pidff` $<1$ in all 30 cells per law |
| H6 | At the nominal vehicle the swing laws cost tracking relative to arm 1 (the direction of the notebook-06 grid at this condition) | `c_T_off_vs_pidff` $>1$ for the softer law and the damper in both scenarios |

## 3. Experiment identity

In [1]:
%matplotlib inline
import math, itertools, time
from functools import lru_cache
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display

from mrac_lab import provenance, style, plots, widgets, mismatch_grid

prov = provenance.setup(pinned_commit="2a329ad")   # commit this notebook was written against (+ W2d, uncommitted at the time)
style.init()

from experiments.mrac.baselines import (
    canonical_offline_params, run_episode, MISMATCH_RANGES, mismatch_cases, VehicleMismatch,
    S30_L_NOM, S30_M_NOM, SOFTER_LAW_GAINS, DAMPER_GAINS,
)
from experiments.mrac.offline_metrics import episode_metrics, c_T_off, c_S_off

CANON = canonical_offline_params()
assert hasattr(CANON, "with_vehicle_mismatch"), (
    "this notebook needs CanonicalOffline.with_vehicle_mismatch (W2d); set MRAC_THESIS_ROOT to a checkout that has it")
VEH, G = CANON.quad, CANON.quad.g
SOURCES = {r.name + " [" + r.unit + "]": "%s  <- %s" % (r.value, r.source) for r in CANON.records}
provenance.banner(prov, "08 Vehicle-mismatch grid for PID+FF and the linear swing laws", SOURCES,
                  notes=list(CANON.caveats) + [
                      "Mismatch (plant only): true m_q, J, a_p scaled by (1 + x); motor effectiveness kappa_L = 1 - delta, kappa_R = 1 + delta. "
                      "Controller and allocator keep m = %.1f kg, J = %.4f kg m^2, a_n = %.3f m, kappa = 1; FF m_hat = 4 kg; payload 4 kg, 2 m." % (CANON.gains.m, CANON.gains.J, CANON.allocator.a_n),
                      "Mismatch ranges (Javier, 2026-11-24): " + "; ".join("%s: %s" % (k, v) for k, v in MISMATCH_RANGES.items()),
                      "Controllers: arm 1 = mass-blind PID+FF (m_hat = 4 kg); softer = + G(-1,-0.5); damper = + G(-1,-2); G(...) = FIXED form kd sqrt(L_nom/L) (at L = L_nom both forms coincide).",
                      "Ratios: *_vs_nom has the SAME controller on the NOMINAL vehicle in the denominator; *_vs_pidff has arm 1 at the SAME mismatch case in the denominator.",
                      "Scenarios: " + "; ".join(f"{k}: {v['text']}" for k, v in mismatch_grid.SCENARIOS.items()),
                      "Thesis checkout is the W2d worktree while the API is uncommitted (MRAC_THESIS_ROOT); after integration the default root works."])

08 Vehicle-mismatch grid for PID+FF and the linear swing laws
Label: EXPLORATORY / offline / PROXY (development PC, reduced taut point-load model)
Thesis checkout: C:\Users\Jcardenasbohorquez\OneDrive - Expleo France\Dokumente\Personal\thesis\gym-pymunk-quadrotor-suspended-load
Commit: 2a329ad957367fc1bc2613f268b1c37b040b7d48 (branch Claude, WORKING TREE HAS UNCOMMITTED CHANGES under experiments/ or gym_pymunk_quadrotor/)
Python: 3.12.3
Claim limits: No stability claim, no Pymunk-transfer claim, no hardware claim. Offline metrics are named *_off and are not the canonical c_T / c_S.
import origin: experiments.mrac.quad -> C:\Users\Jcardenasbohorquez\OneDrive - Expleo France\Dokumente\Personal\thesis\gym-pymunk-quadrotor-suspended-load\experiments\mrac\quad.py
import origin: experiments.mrac.quad_load_pend -> C:\Users\Jcardenasbohorquez\OneDrive - Expleo France\Dokumente\Personal\thesis\gym-pymunk-quadrotor-suspended-load\experiments\mrac\quad_load_pend.py
import origin: experiments.note

m_q [kg],9.6 <- gym_pymunk_quadrotor/plant/config.py:61
width [m],1.164 <- gym_pymunk_quadrotor/plant/config.py:62
height [m],0.759 <- gym_pymunk_quadrotor/plant/config.py:63
J [kg m^2],1.5447816 <- gym_pymunk_quadrotor/plant/entities/quadrotor.py:94-95 (m (w^2+h^2)/12)
a_p = a_n [m],"0.582 <- gym_pymunk_quadrotor/plant/entities/quadrotor.py:96-97,301 (width/2)"
vehicle max_thrust (total) [N],"201.036 <- gym_pymunk_quadrotor/plant/entities/quadrotor.py:157-159 ((m+max_payload) g + m max_ascent_speed; config.py:66,69)"
T_max (per motor) [N],100.518 <- gym_pymunk_quadrotor/plant/entities/quadrotor.py:300; plant/entities/motor.py:135 (thrust = max_thrust_per_motor cmd^2)
tau_m [s],0.05 <- gym_pymunk_quadrotor/plant/config.py:72; motor.py:92-93 (first-order lag)
g [m/s^2],9.81 <- gym_pymunk_quadrotor/config/physics_legacy.py:25
kappa_L = kappa_R [-],1.0 <- no effectiveness split in Pymunk
d [m],"0.3795 <- gym_pymunk_quadrotor/plant/xpbd_cable.py:330-333 (pivot_local = (0, -height/2): cable leaves the bottom edge)"


## 4. Theory

### 4.1 What is mismatched and what is not

The plant is `QuadLoadPend` with the **true** vehicle; the controller (PID, feedforward, swing law) and the nominal allocator keep the **design** vehicle. The load is the nominal one ($m_L=4$ kg, $l=2$ m, $d=0.3795$ m) in every case. One parameter changes at a time.

| Parameter | True plant | Controller / allocator keeps | Expected mechanism (INFERRED from the structure of the laws, not derived here) |
|---|---|---|---|
| $m_q$ (range $0,\pm10,\pm20$ %) | $m_q(1+x)$ | PID mass $m=9.6$ kg in $u_{\rm PID}=m\,a_{d,z}/\cos\theta$ and in the tilt map; $\hat m=4$ kg in the FF | static vertical force error $(m_q^{true}-m)\,g$ until the integrator absorbs it: a height transient, either sign |
| $J$ (range $0,\pm15,\pm30$ %) | $J(1+x)$ | $J$ in $\tau_{\rm PID}=-J[k_{p,\theta}(\theta-\theta_d)+\dots]$ | realised angular acceleration is $(J_{nom}/J^{true})$ times the designed one: the attitude loop gain scales; no effect when the attitude is not excited |
| $a_p$ (range $0,\pm5,\pm10$ %) | $a_p(1+x)$ | allocator arm $a_n=0.582$ m | realised torque is $a_p/a_n$ times the request (gain error in the same attitude loop); the damper torque scales the same way |
| $\delta$ (range $0,0.05,0.10$) | $\kappa_L=1-\delta,\ \kappa_R=1+\delta$ | allocator assumes $\kappa=1$ | an equal thrust split gives a spurious torque $a_p(\kappa_R-\kappa_L)T_i=2\delta\,a_pT_i$, with $T_i\approx(m_q+m_L)g/2$; the attitude PD has no torque integrator |

The cell below prints the hand estimate of the $\delta$ torque bias at hover. The motor states of the initial hover are the **exact equilibrium of the true vehicle** (each motor delivers half of the weight of vehicle plus load, so under asymmetry the left and right motor states differ); a response is therefore the effect of the control law meeting the mismatch, not a start from an arbitrary initial state. A mismatched vehicle is still at rest at tick 0, but the controller's symmetric allocation is not the equilibrium split.

### 4.2 Metrics and ratios

`pos_rms_m` is the vehicle-to-reference position RMS (m); `swing_rms_deg` is the cable-angle RMS about $\phi_{eq}$ (deg); `peak_T_over_Tnom` the peak cable tension over $m_Lg$. Ratios (offline, not the canonical $c_T$, $c_S$):

* `c_T_off_vs_nom`, `c_S_off_vs_nom`: this controller on the mismatched vehicle over the **same controller on the nominal vehicle** (1 = no change). A controller with a worse nominal tracking is flattered by this ratio; the absolute `pos_rms_m` is in section 7.
* `c_T_off_vs_pidff`, `c_S_off_vs_pidff`: this controller over **arm 1 at the same mismatch case** (below 1 = better than the mass-blind PID+FF under that mismatch).

A ratio is blank (NaN) if either episode failed; no failure is expected.

### 4.3 Scope

15 cases (nominal, $m_q$ 4 values, $J$ 4, $a_p$ 4, $\delta$ 2) $\times$ 3 controllers $\times$ 2 scenarios (hover recovery from $\phi_0=0.35$ rad; 4 m min-jerk transfer) = 90 deterministic episodes, one run each, cached in `results/vehicle_mismatch_grid.csv` with a provenance header. No combinations of mismatches, other loads or lengths, wind, kicks or seeds.

In [2]:
# hand estimate of the delta torque bias at hover (arithmetic from the canonical values; compare with the observations)
T_i = (VEH.m + CANON.payload.m_L) * G / 2.0                    # N per motor at hover
for dl in (0.05, 0.10):
    bias = 2.0 * dl * VEH.a_p * T_i
    print("delta = %.2f: spurious torque 2 delta a_p T_i = %.2f N m (tau_max = %.1f N m); attitude-PD offset bias/(J kp_theta) = %.3f rad = %.1f deg (before the outer loop reacts)" % (
        dl, bias, CANON.tau_max, bias / (CANON.gains.J * CANON.gains.kp_theta), math.degrees(bias / (CANON.gains.J * CANON.gains.kp_theta))))

delta = 0.05: spurious torque 2 delta a_p T_i = 3.88 N m (tau_max = 57.9 N m); attitude-PD offset bias/(J kp_theta) = 0.101 rad = 5.8 deg (before the outer loop reacts)
delta = 0.10: spurious torque 2 delta a_p T_i = 7.76 N m (tau_max = 57.9 N m); attitude-PD offset bias/(J kp_theta) = 0.201 rad = 11.5 deg (before the outer loop reacts)


## 5. Interactive exploration

### 5.1 The cached grid

Computed once by `python -m mrac_lab.make_mismatch_grid` (a module-level worker, `ProcessPoolExecutor(max_workers=6)`, about 40 s for the 90 episodes) and loaded here; `RECOMPUTE = True` reruns it and overwrites the CSV.

In [3]:
RECOMPUTE = False          # True: rerun the 90 episodes (about 1 min with 6 worker processes) and overwrite the CSV
CSV = Path("results/vehicle_mismatch_grid.csv")
if RECOMPUTE or not CSV.exists():
    t0 = time.time()
    GRID = mismatch_grid.run_grid(prov.root, workers=6)
    mismatch_grid.save_csv(GRID, CSV, prov, CANON, 6, time.time() - t0)
    GRID_HEADER = ["recomputed in this session"]
else:
    GRID, GRID_HEADER = mismatch_grid.load_csv(CSV)
print("\n".join(GRID_HEADER))
print("rows: %d; failures: %d" % (len(GRID), int(GRID.failed.sum())))
labels_now = [c.mismatch.label for c in mismatch_cases()]
assert list(GRID.sort_values("case").drop_duplicates("case").label) == labels_now, "cached cases differ from experiments.mrac.baselines.mismatch_cases()"
header_commit = next((ln.split()[1] for ln in GRID_HEADER if ln.startswith("thesis_commit:")), "?")
if not prov.commit.startswith(header_commit[:7]):
    print("NOTE: the cache was computed at thesis commit %s, this kernel is at %s." % (header_commit[:7], prov.commit[:7]))
if GRID.failed.any():
    display(GRID[GRID.failed][["scenario", "label", "arm", "failure"]].style.hide(axis="index"))
ARM_LABEL = {"pid_ff": "arm 1: PID+FF (blind FF)", "softer": "softer G(-1,-0.5)", "damper": "damper G(-1,-2)"}
CASE_LABELS = labels_now
FAMILIES = [("m_q", "vehicle mass $m_q$ change [%]"), ("J", "inertia $J$ change [%]"), ("a_p", "true arm $a_p$ change [%]"), ("delta", r"motor asymmetry $\delta$ [%] ($\kappa_{L,R}=1\mp\delta$)")]

EXPLORATORY / offline / PROXY: offline vehicle-mismatch grid, one deterministic run per cell
created: 2026-11-24T21:14:59  workers=6  wall_s=50
thesis_commit: 2a329ad957367fc1bc2613f268b1c37b040b7d48  branch: Claude  dirty: True
thesis_root: C:\Users\Jcardenasbohorquez\OneDrive - Expleo France\Dokumente\Personal\thesis\gym-pymunk-quadrotor-suspended-load
timing: T_s=0.02 N=10 sim_dt=0.002
nominal vehicle: m=9.6 J=1.5447816 a_p=0.582 T_max=100.518 tau_m=0.05 d=0.3795; a_n=0.582
gains: PIDGains(kp_y=2.2, kp_z=5.3, kd_y=2.0, kd_z=3.0, kp_theta=25.0, kd_theta=20.0, ki_y=0.6, ki_z=1.8, m=9.6, J=1.5447816, g=9.81, min_thrust=2.0103600000000004, max_thrust=201.036, max_torque=57.91646124, max_pitch_angle=0.6, beta_smoothing=0.0)
beta_tau=0.3 tau_max=57.91646124 L_nom=2.0 m_L_nom=4.0; load m_L=4.0 L=2.0
mismatch ranges (fractions; delta is kappa_L = 1 - delta, kappa_R = 1 + delta): m_q: (-0.2, -0.1, 0.1, 0.2); J: (-0.3, -0.15, 0.15, 0.3); a_p: (-0.1, -0.05, 0.05, 0.1); delta: (0.05, 0.1)
contr

### 5.2 Response to each mismatch family (interactive)

For the chosen scenario and metric: one panel per family, the nominal vehicle at 0, the three controllers as lines. `c_*_vs_nom` panels show the change against the same controller on the nominal vehicle; `c_*_vs_pidff` panels show the swing law against arm 1 at the same case; the absolute metrics show what the ratios hide.

In [4]:
METRICS = {
    "c_T_off_vs_nom": "tracking ratio vs the nominal vehicle, same controller (1 = no change)",
    "c_S_off_vs_nom": "swing ratio vs the nominal vehicle, same controller (1 = no change)",
    "c_T_off_vs_pidff": "tracking ratio vs arm 1 at the same case",
    "c_S_off_vs_pidff": "swing ratio vs arm 1 at the same case",
    "pos_rms_m": "position RMS [m] (absolute)",
    "swing_rms_deg": "swing RMS about phi_eq [deg] (absolute)",
    "peak_T_over_Tnom": "peak tension / (m_L g)",
    "final_pos_err_m": "final position error [m]",
    "peak_abs_theta_deg": "peak |theta| [deg]",
}

def draw_response(scenario="hover", metric="c_T_off_vs_nom", logy=False, pid_ff=True, softer=True, damper=True):
    d = GRID[GRID.scenario == scenario]
    arms = [a for a, on in (("pid_ff", pid_ff), ("softer", softer), ("damper", damper)) if on]
    fig, ax = plt.subplots(1, 4, figsize=(16, 3.8), sharey=True)
    for a_, (fam, xl) in zip(ax, FAMILIES):
        sub = d[(d.parameter == fam) | (d.case == 0)]
        for arm in arms:
            s_ = sub[sub.arm == arm].assign(x=lambda t: 100.0 * t.value).sort_values("x")
            a_.plot(s_.x, s_[metric], marker="o", ms=4, **style.arm_style(arm))
        a_.axvline(0.0, color="0.75", lw=0.6)
        if metric.startswith("c_"):
            a_.axhline(1.0, color="0.5", lw=0.7, ls=":")
        if logy:
            a_.set_yscale("log")
        a_.set_xlabel(xl, fontsize=8); a_.grid(True, alpha=0.3)
    ax[0].set_ylabel(metric); ax[0].legend(fontsize=7)
    fig.suptitle("EXPLORATORY / offline / PROXY. %s: %s. One deterministic run per point; nominal vehicle at 0; plant-only mismatch at the nominal load (4 kg, 2 m)" % (
        mismatch_grid.SCENARIOS[scenario]["label"], METRICS[metric]), fontsize=8)
    fig.tight_layout(); return fig

panel_response = widgets.param_panel({
    "scenario": ("dropdown", list(mismatch_grid.SCENARIOS), "hover", "scenario"),
    "metric": ("dropdown", list(METRICS), "c_T_off_vs_nom", "metric"),
    "logy": ("check", False, "log y axis"),
    "pid_ff": ("check", True, "arm 1 (blind FF)"),
    "softer": ("check", True, "softer G(-1,-0.5)"),
    "damper": ("check", True, "damper G(-1,-2)"),
}, draw_response)

### 5.3 One episode under mismatch (interactive)

The six standard panels for the chosen mismatch case and scenario (`run_episode` live, cached per case). The grey solid line, when switched on, is the **nominal-vehicle** run of the first selected controller, so the effect of the mismatch can be read against it.

In [5]:
def to_record(r):
    return dict(t=r.t, p=r.p, ref_p=r.ref_p, theta=r.theta, phi=r.phi, phi_dot=r.phi_dot, phi_eq=r.phi_eq, T=r.T,
                f_req=r.f_req, tau_req=r.tau_req, f_alloc=r.f_alloc, tau_alloc=r.tau_alloc, sat=r.saturated, failure=r.failure)

@lru_cache(maxsize=256)
def run_cell(case, arm, scenario):
    cs = mismatch_cases()[case]
    plant, ctrl, ref, ic, n, canon = mismatch_grid.build_case(CANON, cs.mismatch, arm, scenario)
    return run_episode(plant, ctrl, ref, canon.timing, n, ic)

def draw_explorer(case_label="delta=0.10", scenario="hover", pid_ff=True, softer=False, damper=True, overlay_nominal=True):
    case = CASE_LABELS.index(case_label)
    arms = [a for a, on in (("pid_ff", pid_ff), ("softer", softer), ("damper", damper)) if on]
    if not arms:
        arms = ["pid_ff"]
    results = {a: run_cell(case, a, scenario) for a in arms}
    runs = {a: to_record(r) for a, r in results.items()}
    ref_rec = next(iter(runs.values()))
    style.LABEL_OVERRIDE.clear()
    if overlay_nominal and case != 0:
        runs["ff_true"] = to_record(run_cell(0, arms[0], scenario))
        style.LABEL_OVERRIDE["ff_true"] = "nominal vehicle, same controller (%s)" % ARM_LABEL[arms[0]]
    fig = plots.standard_figure(runs, T_nom=CANON.T_nom, ref=ref_rec,
                                title="mismatch: %s, %s" % (case_label, mismatch_grid.SCENARIOS[scenario]["label"]))
    style.LABEL_OVERRIDE.clear()
    msgs = [f"{a}: {r.failure.kind} at {r.failure.time:.2f} s" for a, r in results.items() if r.failed]
    if msgs:
        fig.text(0.5, 0.005, "FAILURE (episode ended): " + "; ".join(msgs), ha="center", color=style.SLACK_COLOR, fontsize=8)
    return fig

panel_explorer = widgets.param_panel({
    "case_label": ("dropdown", CASE_LABELS, "delta=0.10", "mismatch case"),
    "scenario": ("dropdown", list(mismatch_grid.SCENARIOS), "hover", "scenario"),
    "pid_ff": ("check", True, "arm 1 (blind FF)"),
    "softer": ("check", False, "softer G(-1,-0.5)"),
    "damper": ("check", True, "damper G(-1,-2)"),
    "overlay_nominal": ("check", True, "overlay nominal vehicle (first selected controller)"),
}, draw_explorer)

## 6. Observations

In [6]:
obs = []
def add(tag, text, value, ok=None):
    obs.append(dict(id=tag, observation=text, value=value, consistent_with_prediction=ok))

d = GRID
SCEN, ARMS = list(mismatch_grid.SCENARIOS), ["pid_ff", "softer", "damper"]

# H1: identity and the cross-check against notebook 06's grid at (2 m, 4 kg)
nom = d[d.case == 0]
dev = float(np.nanmax(np.abs(nom[["c_T_off_vs_nom", "c_S_off_vs_nom"]].to_numpy() - 1.0)))
add("O1 (H1)", "nominal case: max |ratio vs nominal vehicle - 1|", "%.1e" % dev, dev == 0.0)
try:
    OLD, _ = mismatch_grid.load_csv("results/s30_offline_grid.csv")
    old = OLD[(OLD.L == 2.0) & (OLD.m_L == 4.0) & OLD.arm.isin(ARMS)]
    mm = nom.merge(old, on=["scenario", "arm"], suffixes=("", "_nb06"))
    diff = float(max((mm.pos_rms_m - mm.pos_rms_m_nb06).abs().max(), (mm.swing_rms_deg - mm.swing_rms_deg_nb06).abs().max()))
    add("O2 (H1)", "nominal case against the notebook-06 grid at (L, m_L) = (2 m, 4 kg), 6 cells: max |difference| of position RMS and swing RMS", "%.1e" % diff, diff < 1e-9)
except FileNotFoundError:
    add("O2 (H1)", "notebook-06 grid cache not found", "NOT RUN", None)
add("O3", "failed episodes (of %d)" % len(d), int(d.failed.sum()), None)

# H2: worst c_T_off_vs_nom per family
worst_T = d[d.case > 0].groupby(["scenario", "arm", "parameter"]).c_T_off_vs_nom.max().unstack()
top = worst_T.idxmax(axis=1)
add("O4 (H2)", "family with the largest worst-case c_T_off_vs_nom, per (scenario, controller)", "; ".join("%s/%s: %s" % (a, b, v) for (a, b), v in top.items()), bool((top == "delta").all()))
dlt = d[d.parameter == "delta"]
mono = all(bool(g.sort_values("value").c_T_off_vs_nom.is_monotonic_increasing) and bool((g.c_T_off_vs_nom > 1).all()) for _, g in dlt.groupby(["scenario", "arm"]))
add("O5 (H2)", "delta: c_T_off_vs_nom above 1 and increasing with delta in every (scenario, controller)", str(mono), mono)

# H3: vehicle mass
mq = d[d.parameter == "m_q"]
above = bool((mq.c_T_off_vs_nom > 1).all())
inc = all(float(g[g.value == -0.20].c_T_off_vs_nom.iloc[0]) > float(g[g.value == -0.10].c_T_off_vs_nom.iloc[0]) and
          float(g[g.value == 0.20].c_T_off_vs_nom.iloc[0]) > float(g[g.value == 0.10].c_T_off_vs_nom.iloc[0]) for _, g in mq.groupby(["scenario", "arm"]))
add("O6 (H3)", "m_q: all 24 cells have c_T_off_vs_nom > 1; larger at 20 % than at 10 % on each side", "above 1: %s; 20 %% worse than 10 %%: %s" % (above, inc), above and inc)
side = mq.pivot_table(index=["scenario", "arm"], columns="value", values="c_T_off_vs_nom")
add("O7", "m_q: lighter (-20 %) against heavier (+20 %) c_T_off_vs_nom, fraction of the six (scenario, controller) pairs where the lighter vehicle is worse", "%d of 6" % int((side[-0.20] > side[0.20]).sum()), None)

# H4: J and a_p small
small = d[d.parameter.isin(["J", "a_p"])].assign(dev=lambda t: (t.c_T_off_vs_nom - 1).abs()).groupby(["scenario", "arm"]).dev.max()
big = d[((d.parameter == "m_q") & (d.value.abs() == 0.10)) | ((d.parameter == "delta") & (d.value == 0.05))].assign(dev=lambda t: (t.c_T_off_vs_nom - 1).abs()).groupby(["scenario", "arm"]).dev.min()
add("O8 (H4)", "per (scenario, controller): max |c_T_off_vs_nom - 1| over the J and a_p cases against the smallest over the m_q = +-10 % and delta = 0.05 cases",
    "; ".join("%s/%s: %.4f against %.4f" % (sc_, ar_, small[(sc_, ar_)], big[(sc_, ar_)]) for sc_, ar_ in small.index), bool((small < big).all()))
sw = d[d.case > 0].groupby("parameter").c_S_off_vs_nom.agg(["min", "max"])
add("O9", "c_S_off_vs_nom range over each family (min .. max over the 6 pairs and the family's cases)", "; ".join("%s: %.3f .. %.3f" % (k, r["min"], r["max"]) for k, r in sw.iterrows()), None)

# H5: swing reduction persists
laws = d[d.arm.isin(["softer", "damper"])]
cnt = int((laws.c_S_off_vs_pidff < 1).sum())
add("O10 (H5)", "cells with c_S_off_vs_pidff < 1 for the two swing laws (of %d); largest value" % len(laws), "%d of %d; max %.3f" % (cnt, len(laws), laws.c_S_off_vs_pidff.max()), cnt == len(laws))
hi = laws[laws.case > 0].sort_values("c_S_off_vs_pidff").iloc[-1]
add("O11", "the least swing reduction is at", "%s, %s, %s: %.3f" % (hi.scenario, hi.label, hi.arm, hi.c_S_off_vs_pidff), None)

# H6: tracking cost at the nominal vehicle
nl = nom[nom.arm.isin(["softer", "damper"])]
add("O12 (H6)", "nominal vehicle, c_T_off_vs_pidff of the softer law and the damper in both scenarios", "; ".join("%s/%s %.3f" % (r.scenario, r.arm, r.c_T_off_vs_pidff) for r in nl.itertuples()), bool((nl.c_T_off_vs_pidff > 1).all()))
below = laws[(laws.case > 0) & (laws.c_T_off_vs_pidff < 1)]
add("O13", "mismatch cells where a swing law tracks better than arm 1 (c_T_off_vs_pidff < 1), of %d" % len(laws[laws.case > 0]), "%d (%s)" % (len(below), "; ".join("%s %s %s %.3f" % (r.scenario, r.label, r.arm, r.c_T_off_vs_pidff) for r in below.itertuples())), None)

OBS = pd.DataFrame(obs)
pd.set_option("display.max_colwidth", 400)
display(OBS.style.hide(axis="index").set_properties(**{"text-align": "left", "white-space": "pre-wrap"}))
display(Markdown("Worst-case `c_T_off_vs_nom` per family (maximum over the family's cases):"))
display(worst_T.style.format("{:.3f}"))

id,observation,value,consistent_with_prediction
O1 (H1),nominal case: max |ratio vs nominal vehicle - 1|,0.0e+00,True
O2 (H1),"nominal case against the notebook-06 grid at (L, m_L) = (2 m, 4 kg), 6 cells: max |difference| of position RMS and swing RMS",0.0e+00,True
O3,failed episodes (of 90),0,None
O4 (H2),"family with the largest worst-case c_T_off_vs_nom, per (scenario, controller)",hover/damper: delta; hover/pid_ff: delta; hover/softer: delta; transfer/damper: delta; transfer/pid_ff: delta; transfer/softer: delta,True
O5 (H2),"delta: c_T_off_vs_nom above 1 and increasing with delta in every (scenario, controller)",True,True
O6 (H3),m_q: all 24 cells have c_T_off_vs_nom > 1; larger at 20 % than at 10 % on each side,above 1: True; 20 % worse than 10 %: True,True
O7,"m_q: lighter (-20 %) against heavier (+20 %) c_T_off_vs_nom, fraction of the six (scenario, controller) pairs where the lighter vehicle is worse",0 of 6,None
O8 (H4),"per (scenario, controller): max |c_T_off_vs_nom - 1| over the J and a_p cases against the smallest over the m_q = +-10 % and delta = 0.05 cases",hover/damper: 0.0043 against 0.0050; hover/pid_ff: 0.0161 against 0.7317; hover/softer: 0.0153 against 0.3658; transfer/damper: 0.0048 against 0.1217; transfer/pid_ff: 0.0434 against 0.4498; transfer/softer: 0.0194 against 0.3570,True
O9,c_S_off_vs_nom range over each family (min .. max over the 6 pairs and the family's cases),J: 0.974 .. 1.027; a_p: 0.969 .. 1.039; delta: 0.764 .. 0.958; m_q: 0.928 .. 1.138,None
O10 (H5),cells with c_S_off_vs_pidff < 1 for the two swing laws (of 60); largest value,60 of 60; max 0.950,True


Worst-case `c_T_off_vs_nom` per family (maximum over the family's cases):

## 7. Quantitative table

In [7]:
rows = []
for scen in SCEN:
    for arm in ARMS:
        base = d[(d.scenario == scen) & (d.arm == arm) & (d.case == 0)].iloc[0]
        rows.append(dict(scenario=scen, controller=ARM_LABEL[arm], family="nominal", worst_c_T_off_vs_nom=1.0, worst_c_S_off_vs_nom=1.0,
                         pos_rms_m_nominal=base.pos_rms_m, pos_rms_m_worst=base.pos_rms_m, swing_rms_deg_nominal=base.swing_rms_deg, swing_rms_deg_worst=base.swing_rms_deg))
        for fam, _ in FAMILIES:
            sub = d[(d.scenario == scen) & (d.arm == arm) & (d.parameter == fam)]
            rows.append(dict(scenario=scen, controller=ARM_LABEL[arm], family=fam, worst_c_T_off_vs_nom=sub.c_T_off_vs_nom.max(),
                             worst_c_S_off_vs_nom=sub.c_S_off_vs_nom.max(), pos_rms_m_nominal=base.pos_rms_m, pos_rms_m_worst=sub.pos_rms_m.max(),
                             swing_rms_deg_nominal=base.swing_rms_deg, swing_rms_deg_worst=sub.swing_rms_deg.max()))
QT = pd.DataFrame(rows)
display(Markdown("Worst case over each family's cases (largest ratio, largest absolute RMS). Ratios against the SAME controller on the nominal vehicle; `pos_rms_m` and `swing_rms_deg` are absolute, so a controller that tracks worse at nominal is not flattered. Offline, one deterministic run per cell, EXPLORATORY."))
display(QT.style.format({c: "{:.3f}" for c in QT.columns if c not in ("scenario", "controller", "family")}).hide(axis="index"))
display(Markdown("End-of-range cases (absolute position RMS [m]; ratio against arm 1 at the same case in brackets):"))
END = []
for scen in SCEN:
    for lab in ("m_q+20%", "m_q-20%", "J+30%", "a_p+10%", "delta=0.10"):
        r = {"scenario": scen, "case": lab}
        for arm in ARMS:
            c = d[(d.scenario == scen) & (d.arm == arm) & (d.label == lab)].iloc[0]
            r[ARM_LABEL[arm]] = "%.3f (%.2f)" % (c.pos_rms_m, c.c_T_off_vs_pidff)
        END.append(r)
display(pd.DataFrame(END).style.hide(axis="index"))

Worst case over each family's cases (largest ratio, largest absolute RMS). Ratios against the SAME controller on the nominal vehicle; `pos_rms_m` and `swing_rms_deg` are absolute, so a controller that tracks worse at nominal is not flattered. Offline, one deterministic run per cell, EXPLORATORY.

scenario,controller,family,worst_c_T_off_vs_nom,worst_c_S_off_vs_nom,pos_rms_m_nominal,pos_rms_m_worst,swing_rms_deg_nominal,swing_rms_deg_worst
hover,arm 1: PID+FF (blind FF),nominal,1.000,1.000,0.067,0.067,7.744,7.744
hover,arm 1: PID+FF (blind FF),m_q,3.261,1.030,0.067,0.220,7.744,7.978
hover,arm 1: PID+FF (blind FF),J,1.016,1.005,0.067,0.068,7.744,7.782
hover,arm 1: PID+FF (blind FF),a_p,1.016,1.014,0.067,0.068,7.744,7.850
hover,arm 1: PID+FF (blind FF),delta,7.463,0.867,0.067,0.503,7.744,6.714
hover,"softer G(-1,-0.5)",nominal,1.000,1.000,0.101,0.101,6.534,6.534
hover,"softer G(-1,-0.5)",m_q,2.292,1.024,0.101,0.233,6.534,6.692
hover,"softer G(-1,-0.5)",J,1.001,1.009,0.101,0.102,6.534,6.591
hover,"softer G(-1,-0.5)",a_p,1.013,1.004,0.101,0.103,6.534,6.562
hover,"softer G(-1,-0.5)",delta,5.347,0.907,0.101,0.543,6.534,5.928


End-of-range cases (absolute position RMS [m]; ratio against arm 1 at the same case in brackets):

scenario,case,arm 1: PID+FF (blind FF),"softer G(-1,-0.5)","damper G(-1,-2)"
hover,m_q+20%,0.220 (1.00),0.233 (1.06),0.348 (1.58)
hover,m_q-20%,0.202 (1.00),0.215 (1.06),0.281 (1.39)
hover,J+30%,0.068 (1.00),0.102 (1.48),0.245 (3.58)
hover,a_p+10%,0.067 (1.00),0.103 (1.54),0.243 (3.64)
hover,delta=0.10,0.503 (1.00),0.543 (1.08),0.605 (1.20)
transfer,m_q+20%,0.213 (1.00),0.219 (1.03),0.247 (1.16)
transfer,m_q-20%,0.186 (1.00),0.183 (0.99),0.193 (1.04)
transfer,J+30%,0.080 (1.00),0.082 (1.02),0.118 (1.48)
transfer,a_p+10%,0.076 (1.00),0.084 (1.11),0.119 (1.58)
transfer,delta=0.10,0.443 (1.00),0.457 (1.03),0.465 (1.05)


## 8. Interpretation (EXPLORATORY)

In [8]:
# facts that the interpretation below relies on, computed from the cached grid (each line prints True/False or a number)
def _worst(scen, arm, fam, col="c_T_off_vs_nom"):
    return float(d[(d.scenario == scen) & (d.arm == arm) & (d.parameter == fam)][col].max())
print("arm 1, worst c_T_off_vs_nom per family (hover / transfer):  " + ";  ".join("%s %.2f / %.2f" % (f, _worst("hover", "pid_ff", f), _worst("transfer", "pid_ff", f)) for f, _ in FAMILIES))
print("arm 1, absolute position RMS at the nominal vehicle: hover %.3f m, transfer %.3f m" % tuple(
    float(d[(d.scenario == s_) & (d.arm == "pid_ff") & (d.case == 0)].pos_rms_m.iloc[0]) for s_ in ("hover", "transfer")))
nomrow = d[d.case == 0].pivot_table(index="scenario", columns="arm", values="pos_rms_m")
print("the damper has the largest nominal position RMS of the three controllers in both scenarios: %s" % bool((nomrow.idxmax(axis=1) == "damper").all()))
large = d[d.parameter.isin(["m_q", "delta"]) & (d.value.abs() >= 0.10)]
sm = large.pivot_table(index=["scenario", "label"], columns="arm", values="c_T_off_vs_nom")
print("at m_q = +-10, +-20 %% and delta = 0.10 the damper has the smallest c_T_off_vs_nom of the three controllers in %d of %d (scenario, case) rows" % (int((sm.idxmin(axis=1) == "damper").sum()), len(sm)))
hv = d[(d.scenario == "hover") & (d.arm.isin(["pid_ff", "damper"]))].pivot_table(index="case", columns="arm", values="pos_rms_m")
print("hover: cases (of %d) where the damper has a smaller absolute position RMS than arm 1: %d" % (len(hv), int((hv.damper < hv.pid_ff).sum())))
ds = d[d.parameter == "delta"].pivot_table(index=["scenario", "arm"], columns="value", values="c_S_off_vs_nom")
print("c_S_off_vs_nom decreases from delta = 0.05 to 0.10 in all six (scenario, controller) pairs: %s" % bool((ds[0.10] < ds[0.05]).all()))
print("hover: c_S_off_vs_pidff of the softer law / damper at nominal and at delta = 0.10: %s" % "; ".join(
    "%s %.3f -> %.3f" % (a, float(d[(d.scenario == "hover") & (d.arm == a) & (d.case == 0)].c_S_off_vs_pidff.iloc[0]),
                         float(d[(d.scenario == "hover") & (d.arm == a) & (d.label == "delta=0.10")].c_S_off_vs_pidff.iloc[0])) for a in ("softer", "damper")))

arm 1, worst c_T_off_vs_nom per family (hover / transfer):  m_q 3.26 / 2.74;  J 1.02 / 1.03;  a_p 1.02 / 1.04;  delta 7.46 / 5.70
arm 1, absolute position RMS at the nominal vehicle: hover 0.067 m, transfer 0.078 m
the damper has the largest nominal position RMS of the three controllers in both scenarios: True
at m_q = +-10, +-20 % and delta = 0.10 the damper has the smallest c_T_off_vs_nom of the three controllers in 10 of 10 (scenario, case) rows
hover: cases (of 15) where the damper has a smaller absolute position RMS than arm 1: 0
c_S_off_vs_nom decreases from delta = 0.05 to 0.10 in all six (scenario, controller) pairs: True
hover: c_S_off_vs_pidff of the softer law / damper at nominal and at delta = 0.10: softer 0.844 -> 0.928; damper 0.765 -> 0.950


* **Which parameters matter (offline, this plant, these ranges).** The facts above and the observations of section 6 are what the text rests on: motor asymmetry $\delta$ and vehicle mass $m_q$ change the tracking error of arm 1 by large factors at the ends of their ranges, inertia and arm mismatch in the stated ranges by a few percent. The mechanisms of section 4.1 are consistent with this ($\delta$ adds a torque bias that the nominal allocator and the attitude PD do not remove; a mass error is a static force error that the integrator removes only slowly), but **none of them was isolated** here: that needs a controlled study (for example the same cases with the integrators frozen), which was not run. The hand estimate of the $\delta$ bias in section 4 is arithmetic, not a measurement. H4 holds as an inequality in all six (scenario, controller) pairs (O8), but the margin is thin for the damper in the hover scenario (0.0043 against 0.0050): its tracking is already poor at nominal and barely moves, so that pair is not evidence of a difference between the families.
* **A ratio against the same controller flatters the controllers that track worse at nominal.** The damper has the largest nominal position RMS, so for the large mismatches its `c_T_off_vs_nom` is the smallest of the three controllers in every row counted above, while its absolute RMS stays above arm 1's in the hover scenario (count above). Read the ratios together with the absolute columns of section 7. This grid does **not** show that the swing laws make the tracking more robust.
* **Swing laws under mismatch.** Whether the swing reduction relative to arm 1 persists in every cell is H5 in section 6; the least reduction is listed there, and in the hover scenario the reduction is smaller at $\delta=0.10$ than at the nominal vehicle (numbers above). The swing RMS about $\phi_{eq}$ changes less than the tracking error does and falls with $\delta$ (O9 and the line above); why was **not examined** (INFERRED candidate: the mismatch-driven drift dominates the vehicle motion that excites the cable).
* **What this is not.** One deterministic run per cell, a taut point-load plant, a plant-only mismatch at one load, one scenario each for recovery and transfer. A difference of a few percent is not a finding. No stability, safety or Pymunk statement, and no MRAC arm: this grid is the **baseline** against which arm 3 has to be compared on the same cases once its nominal-payload reference exists.

## 9. Decisions and next steps

### Decisions (Javier, 2026-11-24, after the professor meeting)

| Item | Decision |
|---|---|
| Meeting Q3 | The next comparison is an offline grid of PID+FF (with the swing laws) under vehicle-parameter mismatch: this notebook. |
| Mismatch ranges | Moderate, **one at a time**, at the nominal load 4 kg / 2 m; the controllers keep their nominal values. $m_q$: $0,\pm10,\pm20$ %; $J$: $0,\pm15,\pm30$ %; true arm $a_p$: $0,\pm5,\pm10$ % (the allocator keeps $a_n$); motor asymmetry $\kappa_L=1-\delta$, $\kappa_R=1+\delta$ with $\delta\in\{0,0.05,0.10\}$. |
| Grid scope | 3 controllers (mass-blind PID+FF; + softer law, fixed form; + damper, fixed form) $\times$ 2 scenarios, deterministic, one run per cell, cached CSV. |
| Meeting Q1 | The A0 reference carries the nominal payload (4 kg, 2 m); it enters the comparison only after notebook 07's next step. |

### Next steps

* **Arm 3 on the same grid** once the nominal-payload reference with the softer law is implemented (notebook 07 next step); the cases, scenarios and metrics here are the fixed protocol.
* **For Javier:** whether one-at-a-time cases are enough before any combination (for example $m_q$ with $\delta$), and whether a statistical protocol (several initial conditions, intervals) is wanted before any ratio of this grid is quoted.
* **NOT RUN here:** Pymunk episodes, training, combined mismatches, other loads or lengths, the oracle swing form, MRAC, wind or kicks, any stability analysis, a mechanism study of the $\delta$ and $m_q$ effects.